### Loading the documnets

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import DirectoryLoader

try:
    print("Loading the documnets...")
    doc_loader = DirectoryLoader(
        path='../data/',
        glob='**/*.docx',
        loader_cls=Docx2txtLoader,
        show_progress=True
    )

    pdf_loader = DirectoryLoader(
        path='../data/',
        glob='**/*.pdf',
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )  

except Exception as e:
    print (f"Error while loding documnets {e}")

C:\Users\praju\AppData\Local\Temp\ipykernel_16968\3713041193.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Loading the documnets...


In [3]:
pdf_data = pdf_loader.load() 
doc_data =  doc_loader.load()
print("Documnets loaded succesfully...")
print(f'Length of the loaded PDF documents: {len(pdf_data)}')
print(f'Length of the loaded DOCX documents: {len(doc_data)}') 

100%|██████████| 1/1 [00:00<00:00, 145.11it/s]

Documnets loaded succesfully...
Length of the loaded PDF documents: 1
Length of the loaded DOCX documents: 1


### Creating output structure

In [4]:
from pydantic import BaseModel, Field

class resume(BaseModel):
    ats_score:int = Field(description="Resume overall ATS score")
    grammer_score:int = Field(description="Resume overall Grammer score")
    skills_found : list = Field(description="All the skill found in the resume")
    missing_skills : list = Field(description="Suggest importent skills, besides the skills already exists")
    interview_questions : list = Field(description="Provide a list of interview question based on the resume")
    suggestion : list = Field(description="Suggest improvement for the uploaded resume")

### Loading LLM model and creating Agent

In [8]:
from langchain.messages import SystemMessage 
from langchain.agents import create_agent
from langchain_fireworks import ChatFireworks
from dotenv import load_dotenv
import os 

load_dotenv()
system_msg = SystemMessage(
    """
    you are an ATS + HR expert resume reviewer 
    go trought all the resume in depth and  evaluate ATS score, 
    grammar, skills, missing skills, interview questions, and 
    suggestions based on the resume 
    """)
print('Loading LLM model...')
try:
    llm = ChatFireworks(
        model='accounts/fireworks/models/kimi-k2p5',
        api_key=os.getenv('FIREWORK_API_KEY'),
    )
    llm_info = {
            "name": llm.profile["name"],
            "max_input_tokens": llm.profile["max_input_tokens"],
            "max_output_tokens": llm.profile["max_output_tokens"],
            "release_date": llm.profile['release_date'],
            "last_updated": llm.profile['last_updated']
        }
    agent = create_agent(
        model=llm,
        system_prompt=system_msg,
        response_format=resume
    )
    
    print(f'LLM model loaded: {llm_info}')
    print("Agent created successfully!!!")
except Exception as e:
    print (f'LLM model not found: {e}')

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001F74B7E0640>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001F74B919DE0>


Loading LLM model...
LLM model loaded: {'name': 'Kimi K2.5', 'max_input_tokens': 256000, 'max_output_tokens': 256000, 'release_date': '2026-01-27', 'last_updated': '2026-01-27'}
Agent created successfully!!!


In [20]:
response = agent.stream(
    {
        "message":doc_data[0].page_content
        }
    )

In [21]:
for chunk in response:
    print(chunk)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to act as an ATS + HR expert resume reviewer. However, I notice that no resume has been provided to me. I should ask the user to provide the resume content so I can analyze it properly using the available function.\n\nLet me ask the user to share their resume content.', 'tool_calls': [{'index': 0, 'id': 'functions.resume:0', 'type': 'function', 'function': {'name': 'resume', 'arguments': '{"ats_score": 0, "grammer_score": 0, "skills_found": [], "missing_skills": [], "interview_questions": [], "suggestion": ["Please provide the resume content for analysis."]}'}}]}, response_metadata={'token_usage': {'prompt_tokens': 188, 'total_tokens': 307, 'completion_tokens': 119}, 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'fireworks', 'model_name': 'accounts/fireworks/models/kimi-k2p5'}, id='lc_run--019ef7ef-2f7d-7a93-bf83-a48d297d9b6f-0', tool_c

In [24]:
from langchain_core.prompts import ChatPromptTemplate 

llm_output = llm.with_structured_output(resume)

template = ChatPromptTemplate(
    [
        ('system',"""you are an ATS + HR expert resume reviewer 
    go trought all the resume in depth and  evaluate ATS score, 
    grammar, skills, missing skills, interview questions, and 
    suggestions based on the resume"""),
    ("human", "Hello"),
    ("ai","Hello, How can i help you today!!"),
    ("human","{user_input}")
    ]
)
user_input = doc_data[0].page_content
response = llm_output.invoke(user_input)

In [25]:
for chunk in response:
    print(chunk)

('ats_score', 68)
('grammer_score', 78)
('skills_found', ['Python', 'Java', 'JavaScript', 'Spring Boot', 'Hibernate', 'JPA', 'scikit-learn', 'TensorFlow', 'pandas', 'numpy', 'Git', 'GitHub', 'Postman', 'VS Code', 'Google Colab', 'PostgreSQL', 'MySQL', 'Machine Learning', 'Model Evaluation'])
('missing_skills', ['REST APIs', 'Docker', 'AWS/Cloud Services', 'Linux', 'React/Angular', 'SQL (advanced)', 'Unit Testing (JUnit/PyTest)', 'CI/CD', 'Agile/Scrum', 'Data Structures & Algorithms', 'Microservices', 'MongoDB', 'Redis', 'Kubernetes', 'OAuth/JWT', 'HTML/CSS'])
('interview_questions', ['Explain the machine learning project you worked on for optimizing machine downtime. What algorithms did you use and what were the results?', 'How do you handle database relationships using Hibernate and JPA in Spring Boot?', 'Can you explain the difference between supervised and unsupervised learning with examples from your experience?', 'Describe a challenging bug you encountered and how you debugged it 

In [28]:
user_input = pdf_data[0].page_content
response2 = llm_output.invoke(user_input)

In [30]:
for chunk in response2:
    print(chunk)

('ats_score', 68)
('grammer_score', 72)
('skills_found', ['Python', 'Java', 'Machine Learning', 'Deep Learning', 'XGBoost', 'CNN', 'PyTorch', 'Scikit-Learn', 'FastAPI', 'Numpy', 'Pandas', 'Spring Boot', 'Streamlit', 'PostgreSQL', 'PGvector', 'Git', 'GitHub', 'Docker', 'Google Colab', 'VS Code', 'RAG Embeddings', 'LLM Integration', 'Data Augmentation', 'GPU Acceleration', 'Hyperparameter Tuning', 'Image Classification', 'Fraud Detection', 'Data Preprocessing'])
('missing_skills', ['TensorFlow', 'Keras', 'AWS SageMaker', 'Azure ML', 'MLflow', 'Kubeflow', 'Apache Airflow', 'Spark', 'Hadoop', 'REST API Design', 'Unit Testing', 'CI/CD Pipelines', 'MLOps', 'Feature Engineering', 'Model Monitoring', 'A/B Testing', 'SQL Optimization', 'Kubernetes', 'Linux Shell Scripting', 'Data Visualization (Matplotlib/Seaborn/Plotly)', 'Jupyter Notebooks', 'ONNX', 'Model Quantization', 'Edge Deployment', 'Apache Kafka', 'Redis', 'ElasticSearch', 'Tableau/PowerBI', 'Agile/Scrum'])
('interview_questions', ['E